In [8]:
!pip uninstall -y datasets huggingface_hub -q

!pip install -q \
datasets==3.6.0 \
huggingface_hub==0.34.4 \
transformers==4.55.4 \
accelerate==1.10.1 \
evaluate==0.4.5

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 126.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 101.9 MB/s eta 0:00:00


In [10]:
import torch
import numpy as np

from datasets import load_dataset
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)
import evaluate

In [11]:
import datasets
import huggingface_hub

print("datasets:", datasets.__version__)
print("hub:", huggingface_hub.__version__)

datasets: 5.0.0
hub: 1.20.1


In [13]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")
dataset

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [14]:
tokenizer = BertTokenizer.from_pretrained(
    "bert-base-uncased"
)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [15]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [16]:
tokenized_dataset = tokenized_dataset.remove_columns(
    ["text"]
)

tokenized_dataset = tokenized_dataset.rename_column(
    "label",
    "labels"
)

tokenized_dataset.set_format("torch")

In [17]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [18]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [19]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    return accuracy.compute(
        predictions=predictions,
        references=labels
    )

In [20]:
training_args = TrainingArguments(
    output_dir="./bert_imdb",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=3,

    weight_decay=0.01,

    load_best_model_at_end=True,

    report_to="none"
)

In [22]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [23]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.260230,0.270674,0.920800
2,0.143119,0.268702,0.938040
3,0.067686,0.325157,0.940040


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=9375, training_loss=0.17357359313964843, metrics={'train_runtime': 8844.5499, 'train_samples_per_second': 8.48, 'train_steps_per_second': 1.06, 'total_flos': 1.860035650317888e+16, 'train_loss': 0.17357359313964843, 'epoch': 3.0})

In [24]:
results = trainer.evaluate()

results

Training Loss,Validation Loss,Epoch,Accuracy
0.067686,0.268946,3,0.938120


{'eval_loss': 0.2689460813999176, 'eval_accuracy': 0.93812}

In [25]:
trainer.save_model("./bert_imdb_model")

tokenizer.save_pretrained(
    "./bert_imdb_model"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./bert_imdb_model/tokenizer_config.json', './bert_imdb_model/tokenizer.json')

# app.py

In [27]:
!pip install streamlit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 112.9 MB/s eta 0:00:00


In [29]:
!streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py



2026-06-22 16:14:42.112 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.186.31.143:8501

  Stopping...


In [30]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [31]:
import shutil

# Destination folder in Drive
drive_path = "/content/drive/MyDrive/BERT_Project"

# Copy folders
shutil.copytree("bert_imdb", f"{drive_path}/bert_imdb", dirs_exist_ok=True)
shutil.copytree("bert_imdb_model", f"{drive_path}/bert_imdb_model", dirs_exist_ok=True)

print("Folders copied successfully!")

Folders copied successfully!


In [28]:
import random
import requests
import streamlit as st
import torch

from transformers import (
    BertTokenizer,
    BertForSequenceClassification
)

# -------------------------
# CONFIG
# -------------------------

MODEL_PATH = "bert_imdb_model"
# TMDB_API_KEY = st.secrets["TMDB_API_KEY"]
TMDB_API_KEY = "1ac23e8385ccc53150a6f81284bfa855"

# -------------------------
# LOAD MODEL
# -------------------------

@st.cache_resource
def load_model():
    tokenizer = BertTokenizer.from_pretrained(MODEL_PATH)

    model = BertForSequenceClassification.from_pretrained(
        MODEL_PATH
    )

    model.eval()

    return tokenizer, model


tokenizer, model = load_model()

# -------------------------
# SESSION STATE
# -------------------------

if "current_item" not in st.session_state:
    st.session_state.current_item = None

if "reviews_count" not in st.session_state:
    st.session_state.reviews_count = 0

if "positive_count" not in st.session_state:
    st.session_state.positive_count = 0

if "negative_count" not in st.session_state:
    st.session_state.negative_count = 0

# -------------------------
# TMDB FUNCTIONS
# -------------------------

def get_random_movie():

    page = random.randint(1, 50)

    url = (
        f"https://api.themoviedb.org/3/"
        f"movie/popular"
        f"?api_key={TMDB_API_KEY}"
        f"&page={page}"
    )

    response = requests.get(url).json()

    results = response.get("results", [])

    if not results:
        return None

    return random.choice(results)


def load_new_movie():
    st.session_state.current_item = get_random_movie()


if st.session_state.current_item is None:
    load_new_movie()

movie = st.session_state.current_item

# -------------------------
# UI
# -------------------------

st.title("🎬 BERTTuning")

st.subheader(movie["title"])

col1, col2 = st.columns([1, 2])

with col1:

    if movie.get("poster_path"):
        st.image(
            f"https://image.tmdb.org/t/p/w500"
            f"{movie['poster_path']}"
        )

with col2:

    st.write(
        f"⭐ Rating: {movie['vote_average']:.1f}/10"
    )

    st.write(
        f"📅 Release: {movie['release_date']}"
    )

    st.write(movie["overview"])

# -------------------------
# REVIEW INPUT
# -------------------------

review = st.text_area(
    "Write your review",
    height=180
)

col1, col2 = st.columns(2)

# -------------------------
# SKIP
# -------------------------

with col1:

    if st.button("⏭ Skip"):
        load_new_movie()
        st.rerun()

# -------------------------
# SUBMIT
# -------------------------

with col2:

    if st.button("🚀 Submit Review"):

        if review.strip() == "":
            st.warning(
                "Please write a review first."
            )
            st.stop()

        inputs = tokenizer(
            review,
            truncation=True,
            padding=True,
            max_length=512,
            return_tensors="pt"
        )

        with torch.no_grad():

            outputs = model(**inputs)

            probs = torch.softmax(
                outputs.logits,
                dim=1
            )

            prediction = torch.argmax(
                probs,
                dim=1
            ).item()

        confidence = (
            probs[0][prediction].item()
        )

        labels = {
            0: "Negative 😞",
            1: "Positive 😊"
        }

        st.subheader("Prediction")

        st.success(
            labels[prediction]
        )

        st.write(
            f"Confidence: {confidence:.2%}"
        )

        st.session_state.reviews_count += 1

        if prediction == 1:
            st.session_state.positive_count += 1
        else:
            st.session_state.negative_count += 1

# -------------------------
# STATS
# -------------------------

st.divider()

st.subheader("Session Statistics")

c1, c2, c3 = st.columns(3)

c1.metric(
    "Reviews",
    st.session_state.reviews_count
)

c2.metric(
    "Positive",
    st.session_state.positive_count
)

c3.metric(
    "Negative",
    st.session_state.negative_count
)

# -------------------------
# NEXT MOVIE
# -------------------------

if st.button("🎲 New Random Title"):
    load_new_movie()
    st.rerun()

2026-06-22 16:14:05.765 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

2026-06-22 16:14:06.395 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-22 16:14:06.407 WARNING streamlit.runtime.state.session_state_proxy: Session state does not function when running a script without `streamlit run`
2026-06-22 16:14:06.409 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-22 16:14:06.410 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-22 16:14:06.415 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-22 16:14:06.416 WARNING streamlit.runtime.scriptrunner_utils.script_run_c